<a href="https://colab.research.google.com/github/SreesanthJPN/Alzheimer-s-Disease-Classification-with-Feature-Analysis-Using-Machine-Learning/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
pip install datasets

In [52]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5Tokenizer

class T5RelativePositionEmbedding(nn.Module):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_len = max_len
        self.embedding = nn.Embedding(2 * max_len - 1, embed_dim)

    def forward(self, seq_len, device=None):
        # Create relative position indices
        positions = torch.arange(seq_len, dtype=torch.long, device=device)
        relative_positions = positions[:, None] - positions[None, :]  # Shape: (seq_len, seq_len)
        relative_positions = relative_positions + self.max_len - 1  # Shift to non-negative indices

        # Get relative position embeddings
        pos_emb = self.embedding(relative_positions)  # Shape: (seq_len, seq_len, embed_dim)
        return pos_emb.mean(dim=1)  # Average over the sequence dimension to get (seq_len, embed_dim)


class T5SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_len=512):
        super().__init__()
        self.multi_head_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.pos_embedding = T5RelativePositionEmbedding(max_len, embed_dim)

    def forward(self, x, mask=None):
        seq_len = x.size(0)  # Sequence length
        pos_emb = self.pos_embedding(seq_len)  # Shape: (seq_len, embed_dim)

        # Add positional embeddings to the input
        x = x + pos_emb.unsqueeze(0)  # Add batch dimension to pos_emb

        # Perform multi-head attention
        attn_output, _ = self.multi_head_attn(x, x, x, attn_mask=mask)
        return attn_output

class T5FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        return self.norm(self.fc2(F.gelu(self.fc1(x))))

class T5Block(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, max_len=512):
        super().__init__()
        self.attention = T5SelfAttention(embed_dim, num_heads, max_len)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = T5FeedForward(embed_dim, ff_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))
        x = self.norm2(x + self.ff(x))
        return x

class T5Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.layers = nn.ModuleList([T5Block(embed_dim, num_heads, ff_dim, max_len) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class T5Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.layers = nn.ModuleList([T5Block(embed_dim, num_heads, ff_dim, max_len) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, enc_out, mask=None):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.output_layer(self.norm(x))

class T5Model(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, num_heads=8, ff_dim=2048, num_layers=6, max_len=512):
        super().__init__()
        self.encoder = T5Encoder(vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len)
        self.decoder = T5Decoder(vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, tgt_mask)
        return dec_out

In [53]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Set device

# Load pre-trained T5-large model
pretrained_t5 = T5ForConditionalGeneration.from_pretrained("t5-large")

# Initialize custom T5 model with the same architecture
custom_t5_model = T5Model(
    vocab_size=pretrained_t5.config.vocab_size,
    embed_dim=pretrained_t5.config.d_model,
    num_heads=pretrained_t5.config.num_heads,
    ff_dim=pretrained_t5.config.d_ff,
    num_layers=pretrained_t5.config.num_layers,
    max_len=pretrained_t5.config.max_length
)

# Load weights from pretrained T5-large
custom_t5_model.load_state_dict(pretrained_t5.state_dict(), strict=False)  # strict=False allows partial loading

print("✅ T5-large weights loaded successfully into the custom model!")


✅ T5-large weights loaded successfully into the custom model!


In [55]:
# Set device
device = "cpu"

# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-large")

# Define vocabulary size (should match tokenizer vocab)
vocab_size = tokenizer.vocab_size

# Initialize custom model and move to device
custom_t5_model = T5Model(vocab_size=vocab_size).to(device)

# Test with sample input
input_text = "summarize: The Eiffel Tower is one of the most famous landmarks in the world"

# Tokenize input text & move to device
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Generate output from encoder
with torch.no_grad():
    encoder_outputs = custom_t5_model.encoder(input_ids)

print("Encoder output shape:", encoder_outputs.shape)

Encoder output shape: torch.Size([1, 18, 512])


In [56]:
# Test with sample input
input_text = "summarize: The Eiffel Tower is one of the most famous landmarks in the world"

# Tokenize input text & move to device
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Generate output from encoder
with torch.no_grad():
    encoder_outputs = custom_t5_model.encoder(input_ids)
    print("Encoder output shape:", encoder_outputs.shape)  # Should be (batch_size, seq_len, embed_dim)

    # Generate decoder input (start with the beginning-of-sentence token)
    decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]], device=device)  # Start with pad token
    decoder_outputs = custom_t5_model.decoder(decoder_input_ids, encoder_outputs)
    print("Decoder output shape:", decoder_outputs.shape)  # Should be (batch_size, seq_len, vocab_size)

Encoder output shape: torch.Size([1, 18, 512])
Decoder output shape: torch.Size([1, 1, 32000])


In [57]:
def generate_summary(model, tokenizer, input_text, max_length=50, device="cpu"):
    # Tokenize input text
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

    # Generate encoder outputs
    with torch.no_grad():
        encoder_outputs = model.encoder(input_ids)

        # Initialize decoder input with the beginning-of-sentence token
        decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]], device=device)

        # Iteratively generate tokens
        for _ in range(max_length):
            # Get decoder outputs
            decoder_outputs = model.decoder(decoder_input_ids, encoder_outputs)

            # Get the next token (greedy decoding)
            next_token_id = decoder_outputs[:, -1, :].argmax(dim=-1)

            # Append the next token to the decoder input
            decoder_input_ids = torch.cat([decoder_input_ids, next_token_id.unsqueeze(0)], dim=-1)

            # Stop if the end-of-sentence token is generated
            if next_token_id == tokenizer.eos_token_id:
                break

        # Decode the generated tokens
        summary = tokenizer.decode(decoder_input_ids[0], skip_special_tokens=True)
        return summary

# Test the function
input_text = "summarize: The Eiffel Tower is one of the most famous landmarks in the world"
summary = generate_summary(custom_t5_model, tokenizer, input_text, device=device)
print("Generated summary:", summary)

Generated summary: Tattoo instruction energy Wheel particulière schreibt Gospeltief Alan Yahoo perhapstation déplacementzeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci flag examine liezeci


In [59]:
from datasets import load_dataset

# Load the CNN/DailyMail dataset
ds = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Inspect the dataset
print(ds["train"][0])  # Check the structure of the dataset

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [ ]:
from transformers import T5Tokenizer

# Load the T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Define the preprocessing function
def preprocess_data(examples):
    inputs = ["summarize: " + article for article in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Tokenize the summaries
    labels = tokenizer(examples["highlights"], max_length=150, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# Apply preprocessing to the dataset
tokenized_ds = ds.map(preprocess_data, batched=True)

# Set the dataset format to PyTorch tensors
tokenized_ds.set_format(type="torch", columns=["input_ids", "labels"])

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader

# Create DataLoader for training and validation
train_loader = DataLoader(tokenized_ds["train"], batch_size=8, shuffle=True)
val_loader = DataLoader(tokenized_ds["validation"], batch_size=8)

In [ ]:
import torch
from torch.optim import AdamW

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move the model to the device
custom_t5_model.to(device)

# Define optimizer and loss function
optimizer = AdamW(custom_t5_model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    custom_t5_model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = custom_t5_model(input_ids, labels[:, :-1])
        logits = outputs.view(-1, tokenizer.vocab_size)
        loss = criterion(logits, labels[:, 1:].reshape(-1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader)}")

In [ ]:
custom_t5_model.eval()
total_loss = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = custom_t5_model(input_ids, labels[:, :-1])
        logits = outputs.view(-1, tokenizer.vocab_size)
        loss = criterion(logits, labels[:, 1:].reshape(-1))

        total_loss += loss.item()

print(f"Validation Loss: {total_loss / len(val_loader)}")

In [ ]:
def generate_summary(model, tokenizer, input_text, max_length=150, device="cpu"):
    # Tokenize input text
    input_ids = tokenizer("summarize: " + input_text, return_tensors="pt").input_ids.to(device)

    # Generate encoder outputs
    with torch.no_grad():
        encoder_outputs = model.encoder(input_ids)

        # Initialize decoder input with the beginning-of-sentence token
        decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]], device=device)

        # Iteratively generate tokens
        for _ in range(max_length):
            # Get decoder outputs
            decoder_outputs = model.decoder(decoder_input_ids, encoder_outputs)

            # Get the next token (greedy decoding)
            next_token_id = decoder_outputs[:, -1, :].argmax(dim=-1)

            # Append the next token to the decoder input
            decoder_input_ids = torch.cat([decoder_input_ids, next_token_id.unsqueeze(0)], dim=-1)

            # Stop if the end-of-sentence token is generated
            if next_token_id == tokenizer.eos_token_id:
                break

        # Decode the generated tokens
        summary = tokenizer.decode(decoder_input_ids[0], skip_special_tokens=True)
        return summary

# Test the function
input_text = """
The Hubble Space Telescope has captured stunning images of distant galaxies.
These images provide insights into the formation and evolution of galaxies.
The telescope, launched in 1990, continues to revolutionize our understanding of the universe.
"""
summary = generate_summary(custom_t5_model, tokenizer, input_text, device=device)
print("Generated summary:", summary)